# Stenosis Temporal Model — Evaluation on dataset2_split (test)

Checkpoint: `stenosis_temporal/runs/stenosis_temporal_1/best.pt` (epoch 12)

In [ ]:
import sys, json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, ".")
from stenosis_temporal.config import Config
from stenosis_temporal.dataset import get_dataloader
from stenosis_temporal.model.detector import StenosisTemporalDetector
from stenosis_temporal.evaluate import run_evaluation

## 1. Training history

In [ ]:
metrics_csv = pd.read_csv("stenosis_temporal/runs/stenosis_temporal_1/metrics.csv")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(metrics_csv["epoch"], metrics_csv["val/mAP_50"], label="mAP@0.5")
axes[0].plot(metrics_csv["epoch"], metrics_csv["val/mAP_75"], label="mAP@0.75")
axes[0].plot(metrics_csv["epoch"], metrics_csv["val/mAP_50_95"], label="mAP@0.5:0.95")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("mAP")
axes[0].set_title("Validation mAP")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(metrics_csv["epoch"], metrics_csv["val/F1"], label="F1", color="green")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1")
axes[1].set_title("Validation F1")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(metrics_csv["epoch"], metrics_csv["val/precision"], label="Precision")
axes[2].plot(metrics_csv["epoch"], metrics_csv["val/recall"], label="Recall")
axes[2].set_xlabel("Epoch")
axes[2].set_title("Validation Precision / Recall")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Load model & run evaluation on test set

In [ ]:
cfg = Config()
cfg.data_root = Path("/home/dsa/stenosis/data/dataset2_split")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

loader = get_dataloader("test", cfg, shuffle=False)

model = StenosisTemporalDetector(cfg).to(device)
ckpt = torch.load(
    "stenosis_temporal/runs/stenosis_temporal_1/best.pt",
    map_location=device, weights_only=False,
)
model.load_state_dict(ckpt["model"])
print(f"Loaded best checkpoint (epoch {ckpt['epoch']})")

In [ ]:
metrics = run_evaluation(model, loader, device, cfg)
metrics

## 3. Results summary

In [ ]:
summary = pd.DataFrame([
    {"Metric": k, "Value": f"{v:.4f}" if isinstance(v, float) else str(v)}
    for k, v in metrics.items()
])
summary.style.hide(axis="index")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

bar_metrics = {
    "AP@0.5": metrics["AP@0.5"],
    "AP@0.75": metrics["AP@0.75"],
    "AP@0.5:0.95": metrics["AP@0.5:0.95"],
    "mAR": metrics["mAR"],
    "F1": metrics["F1"],
    "Precision": metrics["precision"],
    "Recall": metrics["recall"],
}

bars = ax.bar(bar_metrics.keys(), bar_metrics.values(), color="steelblue")
ax.set_ylim(0, max(bar_metrics.values()) * 1.3)
ax.set_ylabel("Score")
ax.set_title("Stenosis Temporal — Test Metrics (dataset2_split)")

for bar, val in zip(bars, bar_metrics.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9)

plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Save metrics
out_path = "stenosis_temporal/runs/stenosis_temporal_1/eval_dataset2_split_test.json"
with open(out_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved to {out_path}")

## 4. Comparison with RF-DETR models

Compare Stenosis Temporal with two RF-DETR runs on the same `dataset2_split` test set:
- **RF-DETR (dataset2_augs)** — trained on dataset2_split with augmentations
- **RF-DETR (arcade+dataset2 trainval)** — trained on combined ARCADE trainval + dataset2

In [ ]:
# Load RF-DETR metrics from their metrics.csv files
rfdetr_augs_csv = pd.read_csv("rfdetr_runs/dataset2_augs/metrics.csv")
rfdetr_trainval_csv = pd.read_csv("rfdetr_runs/arcade_dataset2_trainval/metrics.csv")

# Extract best test metrics for each RF-DETR run
def get_best_test_metrics(df):
    test_rows = df.dropna(subset=["test/mAP_50"])
    best = test_rows.loc[test_rows["test/mAP_50"].idxmax()]
    return {
        "AP@0.5": best["test/mAP_50"],
        "AP@0.75": best["test/mAP_75"],
        "AP@0.5:0.95": best["test/mAP_50_95"],
        "mAR": best["test/mAR"],
        "F1": best["test/F1"],
        "Precision": best["test/precision"],
        "Recall": best["test/recall"],
    }

rfdetr_augs_metrics = get_best_test_metrics(rfdetr_augs_csv)
rfdetr_trainval_metrics = get_best_test_metrics(rfdetr_trainval_csv)

temporal_metrics = {
    "AP@0.5": metrics["AP@0.5"],
    "AP@0.75": metrics["AP@0.75"],
    "AP@0.5:0.95": metrics["AP@0.5:0.95"],
    "mAR": metrics["mAR"],
    "F1": metrics["F1"],
    "Precision": metrics["precision"],
    "Recall": metrics["recall"],
}

# Build comparison table
comparison = pd.DataFrame({
    "Metric": list(temporal_metrics.keys()),
    "Stenosis Temporal": [f"{v:.4f}" for v in temporal_metrics.values()],
    "RF-DETR (dataset2_augs)": [f"{v:.4f}" for v in rfdetr_augs_metrics.values()],
    "RF-DETR (arcade+ds2 trainval)": [f"{v:.4f}" for v in rfdetr_trainval_metrics.values()],
})
comparison.style.hide(axis="index")

In [ ]:
# Grouped bar chart comparison
fig, ax = plt.subplots(figsize=(12, 5))

metric_names = list(temporal_metrics.keys())
x = np.arange(len(metric_names))
width = 0.25

vals_temporal = list(temporal_metrics.values())
vals_augs = list(rfdetr_augs_metrics.values())
vals_trainval = list(rfdetr_trainval_metrics.values())

bars1 = ax.bar(x - width, vals_temporal, width, label="Stenosis Temporal", color="#4C72B0")
bars2 = ax.bar(x, vals_augs, width, label="RF-DETR (dataset2_augs)", color="#DD8452")
bars3 = ax.bar(x + width, vals_trainval, width, label="RF-DETR (arcade+ds2 trainval)", color="#55A868")

ax.set_ylabel("Score")
ax.set_title("Model Comparison — dataset2_split test set")
ax.set_xticks(x)
ax.set_xticklabels(metric_names, rotation=30, ha="right")
ax.legend()
ax.grid(True, alpha=0.2, axis="y")

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.003,
                f"{h:.3f}", ha="center", va="bottom", fontsize=7)

ymax = max(max(vals_temporal), max(vals_augs), max(vals_trainval)) * 1.25
ax.set_ylim(0, ymax)
plt.tight_layout()
plt.show()